# Exploring the NSCLC clustering analysis

Run this notebook from the repository or notebooks folder. It uses the sample-matched data and the same functions as the main analysis.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd() if (Path.cwd()/'src').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT/'src'))
from data_io import aligned_rows
rows = [r for r in aligned_rows(ROOT/'data/source') if r['sample_type']=='01']
print('Primary tumors:', len(rows))
print({k: rows[0][k] for k in ['sample_id','patient_id','MCM5','mir885_activity_proxy']})


## 1. What do the columns mean?

`mir885_activity_proxy` is the negative of a target-gene score, not a direct assay. MCM5 is in that target signature. Why does that complicate interpreting the correlation?

In [ ]:
import numpy as np
X = np.array([[r['MCM5'],r['mir885_activity_proxy']] for r in rows])
print('Correlation:', np.corrcoef(X.T)[0,1])
assert len({r['patient_id'] for r in rows}) == len(rows)


## 2. Fit a baseline

Run K-means with two clusters, then try three. Compare the cluster sizes and feature values. Scaling is fitted along with the model.

In [ ]:
from modeling import fit_kmeans
k = 2
model = fit_kmeans(X, k, seed=52)
labels = model.predict(X)
print('Cluster sizes:', np.bincount(labels))
for cluster in range(k):
    print(cluster, 'mean MCM5, proxy:', X[labels==cluster].mean(axis=0))


## 3. Compare the results

The saved run compares three feature sets with a continuous Gaussian reference. Check the size of the differences before interpreting the groups.

In [ ]:
import json
summary = json.loads((ROOT/'results/summary.json').read_text())
for name, experiment in summary['experiments'].items():
    m = experiment['candidate_metrics']
    print(name, {key: round(m[key],3) for key in ['silhouette','null_silhouette_mean','subsample_ari_mean']})


## 4. Inspect the figures

PCA of two variables into two components is a rotation, not dimension reduction. t-SNE is visualization only.

In [ ]:
from IPython.display import SVG, display
display(SVG(filename=str(ROOT/'results/two_marker_selection.svg')))
display(SVG(filename=str(ROOT/'results/pathways_pca.svg')))


## 5. Next experiment

Recompute the target score without MCM5 using the original-study repository. Before running it, write down the expected effect. Then record what changed in the correlation and cluster stability. See `docs/START_HERE.md` for the remaining steps.